# Chương 4 — Mạng nơ-ron hồi quy: BPTT và dự báo đường huyết CGM 30 phút bằng ba cách

In [1]:
# RNN-00 · Thiết lập đường dẫn: mã chương dùng chung gói của tiểu luận (src/dm) và mã kế thừa của nhóm (src/tl01, tl03)
import os, sys, json, time, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""
from pathlib import Path
warnings.filterwarnings("ignore")
TLF = Path.cwd().parent
for p in ["src", "src/tl01", "src/tl03"]:
    sys.path.insert(0, str(TLF / p))
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
print("Thư mục tiểu luận:", TLF)

Thư mục tiểu luận: /home/ngtukien/Projects/MultiAgent/tieuluan_final


Notebook trình bày chia dữ liệu CGM theo bệnh nhân, kiểm chứng BPTT của RNN viết tay bằng NumPy, so sánh 3 cách cài đặt (NumPy, Keras, PyTorch) cùng LSTM xếp chồng 5 lớp, và đối chiếu với 3 chuỗi tài chính/thương mại.

In [2]:
# RNN-01 · Nạp dữ liệu Shanghai CGM (chia theo bệnh nhân 78/17/17) và tạo đặc trưng [glucose chuẩn hóa, sin(giờ), cos(giờ)]
from dm.ch4_cgm import CGMData
data = CGMData()
for split in ("train", "val", "test"):
    X, H, Y = data.raw[split]
    print(f"{split:5s}: {data.n_patients[split]:2d} bệnh nhân, {len(Y):6,d} cửa sổ 12 bước, glucose TB = {Y.mean():.1f} ± {Y.std():.1f} mg/dL")

train: 78 bệnh nhân, 91,333 cửa sổ 12 bước, glucose TB = 140.4 ± 53.1 mg/dL
val  : 17 bệnh nhân, 16,839 cửa sổ 12 bước, glucose TB = 153.1 ± 59.3 mg/dL
test : 17 bệnh nhân, 18,056 cửa sổ 12 bước, glucose TB = 145.7 ± 53.9 mg/dL


In [3]:
# RNN-02 · Kiểm chứng lan truyền ngược qua thời gian (BPTT) của RNN NumPy trên một cửa sổ CGM
from tl3.ch4_scratch import RNNScratch
Xte, Hte, Yte = data.raw["test"]
Fte = data.features(Xte[:5], Hte[:5])
target = (Yte[:5] - Xte[:5, -1]) / data.sd
rnn = RNNScratch(3, 16, seed=0)
yhat = rnn.forward(Fte[0])
grads = rnn.backward(yhat, float(target[0]))
eps = 1e-5
p0 = rnn.p["Whh"][0, 0]
rnn.p["Whh"][0, 0] = p0 + eps; l1 = 0.5 * (rnn.forward(Fte[0]) - float(target[0])) ** 2
rnn.p["Whh"][0, 0] = p0 - eps; l2 = 0.5 * (rnn.forward(Fte[0]) - float(target[0])) ** 2
rnn.p["Whh"][0, 0] = p0
num_g = (l1 - l2) / (2 * eps)
print(f"Đạo hàm BPTT dL/dWhh[0,0] = {grads['Whh'][0,0]:.6f}, sai phân hữu hạn = {num_g:.6f}, sai lệch = {abs(grads['Whh'][0,0]-num_g):.2e}")

Đạo hàm BPTT dL/dWhh[0,0] = 0.006106, sai phân hữu hạn = 0.006106, sai lệch = 1.71e-12


In [4]:
# RNN-03 · Tổng hợp kết quả dự báo glucose 30 phút của 3 cách cài đặt trên 18.056 cửa sổ kiểm tra
rows = []
for fw in ("numpy", "keras", "torch"):
    d = json.loads((TLF / f"outputs/metrics/ch4_cgm_{fw}.json").read_text())["models"]
    for k, v in d.items():
        rows.append({"framework": fw, "mô hình": k, "RMSE": v["rmse"], "MAE": v["mae"],
                     "MARD (%)": v["mard"], "Clarke A": v["clarke_a"], "tham số": v["params"],
                     "train (s)": v.get("train_s", 0.0)})
pd.DataFrame(rows).round(3)

,framework,mô hình,RMSE,MAE,MARD (%),Clarke A,tham số,train (s)
0,numpy,rnn1,15.024,10.467,7.197,0.951,337,67.291
1,keras,rnn1,14.288,9.995,7.036,0.949,4417,56.693
2,keras,lstm1,13.843,9.613,6.733,0.954,17473,135.328
3,keras,gru1,13.918,9.667,6.775,0.952,13313,97.308
4,keras,lstm5,13.790,9.536,6.672,0.954,149569,1009.249
5,torch,persistence,19.054,12.992,8.824,0.900,0,0.000
6,torch,ridge_ar,14.767,10.312,7.260,0.944,13,0.000
7,torch,rnn1,14.262,10.021,7.075,0.949,4481,32.037
8,torch,lstm1,13.822,9.629,6.754,0.954,17729,22.657
9,torch,gru1,13.910,9.664,6.772,0.953,13313,21.829


In [5]:
# RNN-04 · Đối chiếu trên 3 chuỗi tài chính và thương mại (AAPL, vàng GLD, doanh thu TMĐT)
dom_rows = []
M3 = TLF / "src/tl03/outputs/metrics"
for exp in ("aapl", "gold", "ecommerce"):
    for fw in ("torch", "keras"):
        d = json.loads((M3 / f"ch4_{exp}_{fw}.json").read_text())
        naive_rmse = d["naive"]["test"]["rmse"]
        for m in ("lstm", "gru"):
            t = d[m]["test"]
            dom_rows.append({"chuỗi": exp, "framework": fw, "mô hình": m,
                             "RMSE": t["rmse"], "naive RMSE": naive_rmse,
                             "chênh lệch (%)": 100 * (t["rmse"] - naive_rmse) / naive_rmse,
                             "đúng hướng": t["dir_acc"]})
pd.DataFrame(dom_rows).round(3)

,chuỗi,framework,mô hình,RMSE,naive RMSE,chênh lệch (%),đúng hướng
0,aapl,torch,lstm,2.259,2.259,-0.030,0.520
1,aapl,torch,gru,2.257,2.259,-0.098,0.511
2,aapl,keras,lstm,2.277,2.259,0.771,0.465
3,aapl,keras,gru,2.272,2.259,0.564,0.485
4,gold,torch,lstm,0.738,0.738,-0.007,0.530
5,gold,torch,gru,0.751,0.738,1.775,0.478
6,gold,keras,lstm,0.737,0.738,-0.091,0.502
7,gold,keras,gru,0.747,0.738,1.236,0.474
8,ecommerce,torch,lstm,6835.692,10595.774,-35.487,0.762
9,ecommerce,torch,gru,7415.374,10595.774,-30.016,0.794
